# Stage 9F — One final test

**Goal:** Refit the chosen tree on train plus validation, then evaluate it once on the untouched March–April test period.

**Why:** Validation helped us choose the model and threshold. Test now estimates how that frozen choice works on later requests. We will report overall results and the groups that previously caused trouble. We will not adjust the model or threshold after seeing test scores.

**Current operational decision:** HOLD_FOR_OPERATIONAL_USE. This remains a learning experiment until we review the final results and limitations.

In [ ]:
# Cell 2 — Why: verify the saved choice without opening test rows.
from google.colab import drive
from pathlib import Path
import hashlib, json, pandas as pd, numpy as np, joblib, sklearn, time

drive.mount("/content/drive")
root = Path("/content/drive/MyDrive/nyc311_mlops")
split_id = "20260926T142339294958Z"
experiment_id = "20260926T143147282446Z"
split_dir = root / "splits" / split_id
experiment_dir = root / "experiments" / experiment_id
split = json.loads((split_dir / "manifest.json").read_text())
experiment = json.loads((experiment_dir / "experiment.json").read_text())

if split["split_id"] != split_id or experiment["split_id"] != split_id:
    raise ValueError("Wrong split")
if experiment["experiment_id"] != experiment_id or not split["test_sealed_for_model_selection"]:
    raise ValueError("Wrong experiment or test policy")
if experiment["chosen"]["model"] != "tree" or experiment["chosen"]["threshold"] != 0.2:
    raise ValueError("Unexpected model choice")
if sklearn.__version__ != experiment["versions"]["scikit_learn"]:
    raise ValueError("scikit-learn version differs from the experiment")

chosen_name = experiment["chosen"]["model"]
threshold = experiment["chosen"]["threshold"]
model_file = experiment_dir / experiment["artifacts"][chosen_name]["file"]
if hashlib.sha256(model_file.read_bytes()).hexdigest() != experiment["artifacts"][chosen_name]["sha256"]:
    raise ValueError("Candidate model changed")

print("Frozen choice:", chosen_name, "threshold:", threshold)
print("Split:", split_id, "Experiment:", experiment_id)
print("Test rows remain unopened")

In [ ]:
# Cell 3 — Why: verify and combine train plus validation for final fitting.
frames = {}
for name in ["train", "validation"]:
    file = split_dir / f"{name}.jsonl"
    raw = file.read_bytes()
    if hashlib.sha256(raw).hexdigest() != split["files"][name]["sha256"]:
        raise ValueError(f"{name} changed")
    frame = pd.read_json(file, lines=True)
    if len(frame) != split["files"][name]["rows"]:
        raise ValueError(f"{name} row count changed")
    frames[name] = frame

features = split["features"]
required = set(features) | {"label", "unique_key", "created_date"}
for name, frame in frames.items():
    if set(frame.columns) != required or not frame["label"].isin([0, 1]).all():
        raise ValueError(f"{name} has invalid columns or labels")

fit_rows = pd.concat([frames["train"], frames["validation"]], ignore_index=True)
if fit_rows["unique_key"].duplicated().any():
    raise ValueError("Repeated request ID in fitting rows")
print("Final fitting rows:", len(fit_rows))
print("Test rows remain unopened")

In [ ]:
# Cell 4 — Why: refit the frozen design and record it before opening test.
from sklearn.base import clone
from datetime import datetime, timezone
import platform

final_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
final_dir = root / "final_evaluations" / final_id
final_dir.mkdir(parents=True, exist_ok=False)

candidate = joblib.load(model_file)
final_model = clone(candidate)
started = time.monotonic()
final_model.fit(fit_rows[features], fit_rows["label"])
fit_seconds = round(time.monotonic() - started, 1)
final_model_file = final_dir / "model.joblib"
joblib.dump(final_model, final_model_file)
model_hash = hashlib.sha256(final_model_file.read_bytes()).hexdigest()

locked_plan = {
    "final_id": final_id,
    "split_id": split_id,
    "experiment_id": experiment_id,
    "chosen_model": chosen_name,
    "threshold": threshold,
    "features": features,
    "fit_rows": len(fit_rows),
    "fit_seconds": fit_seconds,
    "model_sha256": model_hash,
    "test_sha256_expected": split["files"]["test"]["sha256"],
    "groups_to_report": ["Snow or Ice", "HEAT/HOT WATER", "BROOKLYN"],
    "metrics_to_report": ["accuracy", "precision", "recall", "f1", "f2"],
    "versions": {"python": platform.python_version(),
                 "pandas": pd.__version__, "numpy": np.__version__,
                 "scikit_learn": sklearn.__version__, "joblib": joblib.__version__},
    "status": "locked_before_test"
}
(final_dir / "locked_plan.json").write_text(
    json.dumps(locked_plan, indent=2) + "\n", encoding="utf-8"
)
print("Final evaluation ID:", final_id)
print("Refit seconds:", fit_seconds)
print("Model and threshold locked; test remains unopened")

In [ ]:
# Cell 5 — Why: open the sealed test once and save every result.
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, fbeta_score

if (final_dir / "test_report.json").exists():
    raise ValueError("This final evaluation already has a test report")
if json.loads((final_dir / "locked_plan.json").read_text()) != locked_plan:
    raise ValueError("Locked plan changed")
if hashlib.sha256(final_model_file.read_bytes()).hexdigest() != model_hash:
    raise ValueError("Final model changed")

test_file = split_dir / "test.jsonl"
raw = test_file.read_bytes()
if hashlib.sha256(raw).hexdigest() != split["files"]["test"]["sha256"]:
    raise ValueError("Test file changed")
test = pd.read_json(test_file, lines=True)
if len(test) != split["files"]["test"]["rows"]:
    raise ValueError("Test row count changed")
if set(test.columns) != required or not test["label"].isin([0, 1]).all():
    raise ValueError("Invalid test rows")
if test["unique_key"].duplicated().any() or test["unique_key"].isin(fit_rows["unique_key"]).any():
    raise ValueError("Repeated request ID across splits")

def score_group(mask, predictions):
    y = test.loc[mask, "label"].to_numpy()
    pred = predictions[mask.to_numpy()]
    late = int(y.sum())
    predicted_late = int(pred.sum())
    return {"rows": int(len(y)), "late_cases": late,
            "predicted_late": predicted_late,
            "recall": float(round(recall_score(y, pred), 3)) if late else None,
            "precision": float(round(precision_score(y, pred), 3)) if predicted_late else None}

probability = final_model.predict_proba(test[features])[:, 1]
prediction = probability >= threshold
y = test["label"].to_numpy()
overall = {
    "accuracy": float(round(accuracy_score(y, prediction), 3)),
    "precision": float(round(precision_score(y, prediction, zero_division=0), 3)),
    "recall": float(round(recall_score(y, prediction, zero_division=0), 3)),
    "f1": float(round(f1_score(y, prediction, zero_division=0), 3)),
    "f2": float(round(fbeta_score(y, prediction, beta=2, zero_division=0), 3)),
}
confusion = {
    "true_late": int(((y == 1) & prediction).sum()),
    "missed_late": int(((y == 1) & ~prediction).sum()),
    "false_alarm": int(((y == 0) & prediction).sum()),
    "correct_on_time": int(((y == 0) & ~prediction).sum()),
}
groups = {
    "Snow or Ice": score_group(test["complaint_type"].eq("Snow or Ice"), prediction),
    "HEAT/HOT WATER": score_group(test["complaint_type"].eq("HEAT/HOT WATER"), prediction),
    "BROOKLYN": score_group(test["borough"].eq("BROOKLYN"), prediction),
}
months = {str(month): score_group(test["month"].eq(month), prediction)
          for month in [3, 4]}
report = {"final_id": final_id, "split_id": split_id, "experiment_id": experiment_id,
          "test_rows": len(test), "threshold": threshold, "overall": overall,
          "confusion": confusion, "groups": groups, "months": months,
          "operational_status": "HOLD_PENDING_REVIEW",
          "lesson_version": "stage9f-v1"}
(final_dir / "test_report.json").write_text(
    json.dumps(report, indent=2) + "\n", encoding="utf-8"
)

print("Final evaluation ID:", final_id)
print("Test rows:", len(test), "Overall:", overall)
print("Confusion counts:", confusion)
print("Groups:", groups)
print("Months:", months)
print("Saved report:", final_dir / "test_report.json")

# Stage 9F takeaway and quiz

**What you learned:** The design and threshold were frozen before the test file was opened. We refit that design on all earlier data, checked file hashes and request IDs, and saved one final test report. We have not changed the model based on test results.

**Easy quiz:**

1. Why can we train the final model on both train and validation now?
2. Why was the threshold written to `locked_plan.json` before test?
3. What does `missed_late` count?
4. If the test results are poor, should we adjust the threshold on this same test period?

Send me the **Overall**, **Confusion counts**, **Groups**, **Months**, and **Final evaluation ID**. We will review the results and make a documented release decision in Stage 9G.